In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# ===============================
# 1️⃣ Spark Session
# ===============================
spark = SparkSession.builder.appName("LateDeliveryLR").getOrCreate()

# ===============================
# 2️⃣ Charger les données
# ===============================
path_data = "./data/processed/DataCoSupplyChain_final.csv"
df = spark.read.csv(path_data, header=True, inferSchema=True)

numeric_cols = [
    "Order Item Quantity",
    "Order Profit Per Order",
    "Product Price",
    "distance_km",
    "order_month",
    "Days for shipment (scheduled)"
]

categorical_cols = [
    "Type",
    "Category Name",
    "Customer Segment",
    "Order Region",
    "Shipping Mode",
    "Market"
]

target_col = "Late_delivery_risk"

# ===============================
# 3️⃣ Encodage des variables
# ===============================
indexers = [StringIndexer(inputCol=c, outputCol=c + "_idx") for c in categorical_cols]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec") for c in categorical_cols]

assembler = VectorAssembler(
    inputCols=[c + "_vec" for c in categorical_cols] + numeric_cols,
    outputCol="features"
)

# ===============================
# 4️⃣ Logistic Regression
# ===============================
lr = LogisticRegression(
    featuresCol="features",
    labelCol=target_col,
    maxIter=50,
    regParam=0.01,
    elasticNetParam=0.0
)

# ===============================
# 5️⃣ Pipeline
# ===============================
pipeline = Pipeline(stages=indexers + encoders + [assembler, lr])

# ===============================
# 6️⃣ Train/Test Split
# ===============================
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# ===============================
# 7️⃣ Train Model
# ===============================
model = pipeline.fit(train_df)

# ===============================
# 8️⃣ Predictions
# ===============================
predictions = model.transform(test_df)
predictions.select(target_col, "prediction", "probability").show(10, truncate=False)

# ===============================
# 9️⃣ Metrics : Accuracy, Precision, Recall, F1
# ===============================

# Accuracy
accuracy_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = accuracy_eval.evaluate(predictions)

# Precision
precision_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="weightedPrecision"
)
precision = precision_eval.evaluate(predictions)

# Recall
recall_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = recall_eval.evaluate(predictions)

# F1-score
f1_eval = MulticlassClassificationEvaluator(
    labelCol=target_col,
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print("\n===== METRICS =====")
print("Accuracy :", accuracy)
print("Precision :", precision)
print("Recall :", recall)
print("F1 Score :", f1)

# ===============================
# 🔟 Save Model
# ===============================
model.write().overwrite().save("./models/logistic_regression_pipeline_late_delivery")

print("\nModèle Logistic Regression sauvegardé.")



+------------------+----------+---------------------------------------+
|Late_delivery_risk|prediction|probability                            |
+------------------+----------+---------------------------------------+
|0                 |1.0       |[0.4327731009801303,0.5672268990198697]|
|0                 |1.0       |[0.42623354815709796,0.573766451842902]|
|0                 |1.0       |[0.4271070039421548,0.5728929960578453]|
|0                 |1.0       |[0.4271070039421548,0.5728929960578453]|
|0                 |1.0       |[0.4327731009801303,0.5672268990198697]|
|0                 |1.0       |[0.4271070039421548,0.5728929960578453]|
|0                 |1.0       |[0.4271070039421548,0.5728929960578453]|
|0                 |1.0       |[0.4324643458205335,0.5675356541794665]|
|0                 |1.0       |[0.4364265918685286,0.5635734081314714]|
|0                 |1.0       |[0.4327731009801303,0.5672268990198697]|
+------------------+----------+---------------------------------

25/11/17 21:28:58 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 17963323 ms exceeds timeout 120000 ms
25/11/17 21:28:58 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/17 22:31:43 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint